# ESCNN Structural Equivariance Experiments
How does ESCNN enforce structural equivariance when it exports a model?

In [ ]:
import torch
import torch.nn as nn

from escnn import gspaces
from escnn import nn as escnn_nn


class EquivariantCNN(torch.nn.Module):

    def __init__(self):
        super().__init__()

        # symmetry group: rotations by multiples of 45 degrees
        self.r2_act = gspaces.rot2dOnR2(N=8)

        # input: scalar field (image)
        in_type = escnn_nn.FieldType(self.r2_act, [self.r2_act.trivial_repr])

        # hidden representation
        hidden_type = escnn_nn.FieldType(
            self.r2_act,
            16 * [self.r2_act.regular_repr]
        )

        out_type = escnn_nn.FieldType(
            self.r2_act,
            32 * [self.r2_act.regular_repr]
        )

        self.block1 = escnn_nn.SequentialModule(
            escnn_nn.R2Conv(in_type, hidden_type, kernel_size=5, padding=2),
            escnn_nn.ReLU(hidden_type),
            escnn_nn.PointwiseMaxPool(hidden_type, 2)
        )

        self.block2 = escnn_nn.SequentialModule(
            escnn_nn.R2Conv(hidden_type, out_type, kernel_size=5, padding=2),
            escnn_nn.ReLU(out_type),
            escnn_nn.PointwiseMaxPool(out_type, 2)
        )

        self.gpool = escnn_nn.GroupPooling(out_type)

        self.fc = nn.Linear(32, 10)

        self.in_type = in_type

    def forward(self, x):

        x = escnn_nn.GeometricTensor(x, self.in_type)

        x = self.block1(x)
        x = self.block2(x)

        x = self.gpool(x)

        x = x.tensor.mean(dim=(2,3))

        return self.fc(x)

In [ ]:
import torchvision.transforms as transforms
import torchvision

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 64

device = "cuda" if torch.cuda.is_available() else "cpu"

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                        shuffle=True, num_workers=2)

transform = transforms.Compose([
    transforms.ToTensor(), 
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                    download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=10000,
                                        shuffle=False, num_workers=2)

In [ ]:
import torch.nn.functional as F

model = EquivariantCNN()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for x, y in trainloader:

    pred = model(x)

    loss = F.cross_entropy(pred, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [ ]:
model.eval()

exported_model = model.export()